In [1]:
#| default_exp xl

In [2]:
#| hide
import nbdev; nbdev.nbdev_export()

In [3]:
!ln -s src rest/src

ln: failed to create symbolic link 'rest/src': File exists


In [4]:
#| export
from os import getenv
model_path = getenv("MODEL")
from rest.gen import generate

In [5]:
model_path = 'xl/pelevin'

In [6]:
#| export
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("./tokenizer/rugpt3xl.tokenizer", local_files_only=True)


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [7]:
#| export
seq_length = 512

from src.xl_wrapper import RuGPT3XL
model = RuGPT3XL.from_pretrained(
    "sberbank-ai/rugpt3xl",
    weights_path=f"./models/{model_path}.model",
    deepspeed_config_path="src/deepspeed_config/gpt3_xl_2048.json",
    seq_len=seq_length,
)
tokenizer = model.tokenizer
model.cuda()
model.eval();

/usr/local/lib/python3.10/dist-packages/_distutils_hack/__init__.py:54: UserWarning: Reliance on distutils from stdlib is deprecated. Users must rely on setuptools to provide the distutils module. Avoid importing distutils or import setuptools first, and avoid setting SETUPTOOLS_USE_DISTUTILS=stdlib. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(


[2024-09-28 13:53:43,717] [INFO] [real_accelerator.py:161:get_accelerator] Setting ds_accelerator to cuda (auto detect)


[W socket.cpp:426] [c10d] The server socket cannot be initialized on [::]:6000 (errno: 97 - Address family not supported by protocol).
[W socket.cpp:653] [c10d] The client socket cannot be initialized to connect to [localhost]:6000 (errno: 97 - Address family not supported by protocol).
[W socket.cpp:653] [c10d] The client socket cannot be initialized to connect to [localhost]:6000 (errno: 97 - Address family not supported by protocol).
/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


> initializing model parallel with size 1


In [8]:
model

RuGPT3XL(
  (model): FP16_Module(
    (module): GPT3Model(
      (word_embeddings): VocabParallelEmbedding()
      (position_embeddings): Embedding(2048, 2048)
      (embedding_dropout): Dropout(p=0.1, inplace=False)
      (transformer): GPT3ParallelTransformer(
        (layers): ModuleList(
          (0-23): 24 x GPT3ParallelTransformerLayer(
            (input_layernorm): FusedLayerNorm(torch.Size([2048]), eps=1e-05, elementwise_affine=True)
            (attention): GPT3ParallelSelfAttention(
              (query_key_value): ColumnParallelLinear()
              (attention_dropout): Dropout(p=0.1, inplace=False)
              (dense): RowParallelLinear()
              (output_dropout): Dropout(p=0.1, inplace=False)
            )
            (post_attention_layernorm): FusedLayerNorm(torch.Size([2048]), eps=1e-05, elementwise_affine=True)
            (mlp): GPT3ParallelMLP(
              (dense_h_to_4h): ColumnParallelLinear()
              (dense_4h_to_h): RowParallelLinear()
        

In [9]:
sum(p.numel() for p in model.parameters())

1315737600

In [10]:
#| export
import deepspeed, torch
ds_engine = deepspeed.init_inference(model,
                                 mp_size=1,
                                 dtype=torch.float16,
                                 checkpoint=None,
                                 replace_with_kernel_inject=True)
model = ds_engine.module

[2024-09-28 13:54:02,869] [INFO] [logging.py:96:log_dist] [Rank -1] DeepSpeed info: version=0.12.6, git-hash=unknown, git-branch=unknown
[2024-09-28 13:54:02,872] [WARNING] [config_utils.py:69:_process_deprecated_field] Config parameter mp_size is deprecated use tensor_parallel.tp_size instead
[2024-09-28 13:54:02,872] [INFO] [logging.py:96:log_dist] [Rank -1] quantize_bits = 8 mlp_extra_grouping = False, quantize_groups = 1


In [11]:
#| export
def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool, temperature:float=1.0):
    return generate(model, tokenizer, seq_length, prompt, length, num_samples, allow_linebreak, temperature)

In [12]:
%%time
get_sample('На словах ты Лев Толстой, а на деле', 50, 4, False)

/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:649: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
/workspaces/gpt/src/xl_wrapper.py:280: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /opt/pytorch/pytorch/torch/csrc/tensor/python_tensor.cpp:83.)
  context_tokens_tensor = torch.cuda.LongTensor(context_tokens)


CPU times: user 5.74 s, sys: 158 ms, total: 5.9 s
Wall time: 5.47 s


[' – уголовник… Ты хоть понимаешь, чем ты рискуешь? Совсем недавно я ловил тебя не за это. Я бы и тебя отпустил, и тех, с кем ты по делу работать будешь.',
 ' говно!" В общем, все как всегда. Гремит, сверкает, мелькают неясные силуэты, страшно и непонятно, кто кого. И только одно утешает: на миг перед глазами возникает то, чем так хочется быть.',
 ' хуй без соли доедаешь… Нет, куда ни шло с Христом, но сейчас? Я тогда зря на тебя свечку ставил? Нечего сказать, золотые руки.',
 ' ты обыкновенный Лев – точнее, как ты недавно со мной поговорил, «Никотиновое животное», потому что все, кто не я, только и могут, что щипаться или цапаться.']

In [14]:
%%time
print(get_sample(' - ты кто?', 50, 1, True)[0])

 - еле выговорил Андрий. - Эс эс эр. Маруся испуганно оглядела перепачканную сестру и поняла, что это она сама и есть. Оказалось, другого ответа и быть не могло.
CPU times: user 1.12 s, sys: 10.4 ms, total: 1.13 s
Wall time: 1.1 s
